In [1]:
# Installation

!pip install unsloth accelerate peft trl bitsandbytes gradio datasets huggingface_hub sentence-transformers faiss-cpu pypdf rouge-score bert-score fpdf langchain-community -q
!pip install --upgrade transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/56.0 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.

In [2]:
# Login to Hugging Face
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# Load base model (4-bit QLoRA)
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

[transformers] Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


In [4]:
# Add LoRA adapters

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing=True,
)

[transformers] Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
#
# Load dataset (customer support Q&A)

from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
dataset = dataset.select(range(1000))  # 1000 examples for speed

README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [6]:

# Format dataset for Llama 3

def format_example(example):
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": prompt}

formatted_dataset = dataset.map(format_example, remove_columns=dataset.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
# Split into train/validation

split_dataset = formatted_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [8]:
# Train with SFTTrainer (500 steps, validation)

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        max_steps=350,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        eval_strategy="steps",  # ✅ changed from evaluation_strategy
        eval_steps=100,
        save_steps=100,
        output_dir="./customer_support_bot",
        load_best_model_at_end=True,
        report_to="none",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/900 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 4 | Total steps = 350
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,0.443803,0.448633
200,0.392281,0.419561
300,0.373878,0.408192
350,0.358037,0.404819


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=350, training_loss=0.5087204224722726, metrics={'train_runtime': 2387.9641, 'train_samples_per_second': 1.173, 'train_steps_per_second': 0.147, 'total_flos': 3.635428814271283e+16, 'train_loss': 0.5087204224722726, 'epoch': 3.097777777777778})

In [9]:
# Save LoRA adapters

model.save_pretrained("lora_customer_bot")
tokenizer.save_pretrained("lora_customer_bot")

[transformers] Unsloth: Restored added_tokens_decoder metadata in lora_customer_bot/tokenizer_config.json.


('lora_customer_bot/tokenizer_config.json',
 'lora_customer_bot/chat_template.jinja',
 'lora_customer_bot/tokenizer.json')

In [10]:
!pip install langchain-text-splitters -q

In [12]:
# Build RAG retriever from the PDF


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load PDF
loader = PyPDFLoader("return_policy_document.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
chunk_texts = [chunk.page_content for chunk in chunks]

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings.astype(np.float32))

def retrieve_context(query, k=3):
    query_emb = embedder.encode([query])
    distances, indices = index.search(query_emb.astype(np.float32), k)
    return [chunk_texts[i] for i in indices[0]]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
# Load fine-tuned model for inference

from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)
fine_tuned_model = PeftModel.from_pretrained(base_model, "lora_customer_bot")
fine_tuned_model.eval()

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [14]:
#  Query function with RAG + robust extraction
#
def ask_bot(user_query):
    context_chunks = retrieve_context(user_query)
    context = "\n\n".join(context_chunks)
    messages = [
        {"role": "system", "content": f"You are a customer support assistant. Use this policy info:\n{context}"},
        {"role": "user", "content": user_query}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to("cuda")
    outputs = fine_tuned_model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

# Test it
print(ask_bot("How many days to return an item?"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/

I understand that you're looking for information on the number of days to return an item. According to our return policy, the standard return window is 30 calendar days from the date of delivery for shipped orders or the date of purchase for in-store transactions. However, please note that different categories may have alternate timelines, such as electronics and devices, which may have a return window of 14 to 30 days. If you have any further questions or concerns, please don't hesitate to reach out to our customer support team.


In [15]:
# Install evaluate library
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/84.1 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [17]:
from evaluate import load
import numpy as np

# Load metrics
bleu = load("bleu")
rouge = load("rouge")
bertscore = load("bertscore")

# Small test set
test_questions = [
    "How do I return a damaged item?",
    "What is your return policy?",
    "Can I return a gift card?",
    "How many days to return?",
    "My order arrived broken, what should I do?"
]

# Reference answers
test_answers = [
    "Contact us within 48 hours for a free replacement or refund.",
    "You have 30 days to return unused items in original packaging.",
    "Gift cards are non-returnable.",
    "30 days from the date you received the item.",
    "Please contact us within 48 hours with your order number and a photo for a free replacement."
]

def evaluate_model():
    predictions = []
    for q in test_questions:
        pred = ask_bot(q)  # uses the fine_tuned_model + RAG
        predictions.append(pred)
        print(f"Q: {q}\nA: {pred}\n")

    # BLEU expects references as list of lists
    bleu_score = bleu.compute(predictions=predictions, references=[[a] for a in test_answers])
    rouge_score = rouge.compute(predictions=predictions, references=test_answers)
    bert = bertscore.compute(predictions=predictions, references=test_answers, lang="en")

    print("\n" + "="*50)
    print("EVALUATION RESULTS")
    print("="*50)
    print(f"BLEU Score: {bleu_score['bleu']:.4f}")
    print(f"ROUGE-L: {rouge_score['rougeL']:.4f}")
    print(f"BERTScore F1: {np.mean(bert['f1']):.4f}")
    print("="*50)

    return {
        "bleu": bleu_score["bleu"],
        "rougeL": rouge_score["rougeL"],
        "bert_f1": np.mean(bert["f1"])
    }

# Run evaluation
results = evaluate_model()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I return a damaged item?
A: I've been informed that you're looking for guidance on returning a damaged item. Our return policy is designed to ensure a seamless process for you. Here's what you need to do:

1. **Initiate the Return**: Reach out to our customer support team through our designated channels, such as our website returns portal, customer service email, chat, in-store desk, or telephone support line. Inform them that you need to initiate a return for a damaged item.
2. **Provide Order Details**: Our support team will ask for the order number associated with the damaged item. Please provide this information to facilitate the process.
3. **Describe the Damage**: When reporting the damage, please provide a detailed description of the issue, including any relevant photographs where available.
4. **Follow Return Instructions**: Our team will guide you through the return process, which may involve shipping the item back to us or arranging for a pickup. Please follow the i

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is your return policy?
A: I'm happy to inform you about our return policy. We strive to ensure that our customers have a seamless experience when it comes to returning items. Here are the key points to keep in mind:

1. Eligibility: To be eligible for a return, the item must meet the following conditions:
	* The return request must be initiated within the applicable return window.
	* The customer must provide acceptable proof of purchase.
	* The item must be returned in its original condition, subject to reasonable inspection.
	* The item must not be excluded under the non-returnable item section.
	* The item must have been purchased through an authorized channel.

2. Return Process: To initiate a return, please follow these steps:
	* Log in to your account on our platform.
	* Navigate to the "Order History" or "My Orders" section.
	* Locate the specific purchase you wish to return.
	* Click on the purchase to access the order details.
	* Look for the "Return Item" option and s

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



EVALUATION RESULTS
BLEU Score: 0.0083
ROUGE-L: 0.1245
BERTScore F1: 0.8503


In [18]:

# Test questions and reference answers (direct quotes from the PDF)
test_questions = [
    "How many days to return an item?",
    "Can I return a gift card?",
    "What should I do if my order arrives damaged?",
    "Are shipping fees refunded?",
    "What items are non-returnable?"
]

test_answers = [
    "The standard return window is 30 calendar days from the date of delivery. Electronics: 14 to 30 days. Damaged or defective items: report within 7 days.",
    "Gift cards and prepaid cards are generally not eligible for return except where required by law or the item arrived damaged, defective, or incorrect.",
    "Contact support promptly with order number, description, and photographs. The seller may offer replacement, exchange, full refund, partial refund, or repair.",
    "Usually only when the seller made an error, the item was defective, or refund of shipping charges is otherwise required by law. Standard outbound shipping charges are often non‑refundable for discretionary returns.",
    "Gift cards, downloadable software, personalized items, final sale merchandise, personal care products after opening, perishable goods, hazardous materials, and items missing original safety seals."
]

# Run evaluation
results = evaluate_model()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How many days to return an item?
A: I understand your question about the number of days to return an item. According to our return policy, the standard return window is 30 calendar days from the date of delivery for shipped orders or the date of purchase for in-store transactions. However, certain categories may have different timelines, such as electronics and devices, which have a return window of 14 to 30 days. It's always best to check the specific return policy for the item you wish to return.



[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Can I return a gift card?
A: I understand your question about returning a gift card. According to our policy, gift cards and prepaid cards are not eligible for returns. Once a gift card is purchased, it is considered a final sale and cannot be returned or exchanged. If you have any further questions or concerns, please don't hesitate to reach out to our customer support team. We're here to assist you during {{Customer Support Hours}} at {{Customer Support Phone Number}} or through the Live Chat on our website at {{Website URL}}.



[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should I do if my order arrives damaged?
A: I've been informed that your order has arrived damaged. To ensure a smooth resolution process, please follow these steps:

1. Contact Support: Reach out to our customer support team promptly, providing them with your order number, a detailed description of the issue, and any relevant photographs where available.
2. Provide Feedback: Our team will guide you through the necessary steps to resolve the problem. They may ask for additional information or clarification to assist in the process.
3. Return the Item: If the damage is due to shipping, you should retain all packaging materials until the claim is resolved. If the damage is with the product itself, please return the item to us in its original condition, along with any original components, such as chargers, cables, inserts, manuals, warranty cards, spare parts, branded packaging, and bonus items bundled with the purchase.
4. Warranty Service: In some cases, our partners may require

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Are shipping fees refunded?
A: I've understood, you're asking if shipping fees are refunded. According to our return policy, shipping fees are typically only refunded in specific circumstances, such as when the seller made an error, the item was defective, or there is a legal requirement to refund shipping charges. In other cases, such as when the customer changes their mind or orders the wrong item, return shipping may be deducted from the refund or charged separately. If a prepaid return label is issued for convenience, the label cost may be deducted from the refund unless the return is due to seller error or product defect.

Q: What items are non-returnable?
A: I've understood you're looking for information on non-returnable items. According to our policy, the following items are generally not eligible for return, except where required by law or where the item arrived damaged, defective, or incorrect:

1. Gift cards and prepaid cards
2. Personalized, customized, or made-to-order 

In [20]:
test_questions = [
    "How many days do I have to return an item?",
    "Can I return a gift card?",
    "What should I do if my order arrives damaged?",
    "Are shipping fees refunded?",
    "What items cannot be returned?"
]

test_answers = [
    "The standard return window is 30 calendar days from the date of delivery. For electronics and devices, the window is 14 to 30 days depending on category. If your item is damaged or defective, you should report it within 7 days of delivery. Commercial or bulk orders may have different terms.",
    "Gift cards and prepaid cards are generally not eligible for return except where required by law or if the item arrived damaged, defective, or incorrect. In most cases, you cannot return a gift card for a refund or exchange.",
    "If your order arrives damaged, please contact support as soon as possible with your order number, a description of the issue, and photographs if available. The seller may offer a replacement, exchange, full refund, partial refund, or repair depending on the situation. Keep all original packaging until the claim is resolved.",
    "Shipping fees are usually refunded only when the seller made an error, the item was defective, or refund of shipping charges is required by law. For discretionary returns (e.g., changing your mind), standard outbound shipping is typically non‑refundable.",
    "Non‑returnable items include gift cards and prepaid cards, downloadable software, personalized or customized items, final sale merchandise, personal care products after opening, perishable goods, hazardous materials, and items missing original safety seals."
]

results = evaluate_model()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
[transforme

Q: How many days do I have to return an item?
A: I've understood you're asking about the number of days you have to return an item. According to our return policy, the standard return window is 30 calendar days from the date of delivery for shipped orders or the date of purchase for in-store transactions. However, certain categories may have different return timelines, such as electronics and devices, which have a return window of 14 to 30 days. If you have any specific questions or concerns about returning an item, please feel free to reach out to our customer support team.



[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Can I return a gift card?
A: I've understood you're wondering if it's possible to return a gift card. According to our policy, gift cards and prepaid cards are not eligible for return or exchange. Once a gift card is purchased, it is considered a final sale and cannot be returned or exchanged for cash or store credit. If you have any further questions or concerns, please don't hesitate to reach out to our customer support team.



[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should I do if my order arrives damaged?
A: I'm on your side that you're facing difficulties with your order arriving damaged. To ensure a smooth resolution, please follow these steps:

1. Contact Support: Reach out to our customer support team promptly, providing them with your order number, a detailed description of the issue, and any supporting photographs where available.
2. Provide Relevant Information: Make sure to provide all necessary details, such as the order number, the nature of the damage, and any relevant photographs or videos.
3. Follow the Recommended Course of Action: Based on the specific circumstances, our support team may recommend one or more remedies, such as replacement, exchange, full or partial refund, or repair.
4. Retain Packaging Materials: Keep all original packaging materials, including boxes, bags, and any inserts, until the claim is resolved. This may be required for inspection or return purposes.
5. Adhere to Return Requirements: When returning 

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Are shipping fees refunded?
A: I've understood, you're asking about the refund of shipping fees. According to our return policy, shipping fees are usually only refunded when the seller made an error, the item was defective, or refund of shipping charges is otherwise required by law. In cases where the customer changes their mind, ordered the wrong item, or no longer wants the product, return shipping may be deducted from the refund or charged separately. However, if a prepaid return label is issued for convenience, the label cost may be deducted from the refund unless the return is due to seller error or product defect.

Q: What items cannot be returned?
A: I understand your question about which items cannot be returned. According to our policy, the following items are generally not eligible for return, except where required by law or where the item arrived damaged, defective, or incorrect:

1. Gift cards and prepaid cards

Please note that there may be specific exceptions or additi

In [19]:
# ============================================================
# CELL 15: Gradio chat interface
# ============================================================
import gradio as gr

def chat(message, history):
    context_chunks = retrieve_context(message)
    context = "\n\n".join(context_chunks)
    messages = [{"role": "system", "content": f"You are a support assistant. Policy:\n{context}"}]
    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to("cuda")
    outputs = fine_tuned_model.generate(**inputs, max_new_tokens=256, temperature=0.3)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

demo = gr.ChatInterface(
    fn=chat,
    title="🤖 Customer Support Bot (Fine-tuned + RAG)",
    description="Knows return policies from PDF + fine-tuned on real Q&A",
    examples=["How many days to return?", "What items cannot be returned?", "My coffee maker arrived broken"]
)
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2242632c05ccda8e91.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
